In [ ]:
# ------------------------------------
# OLD CODE
# ------------------------------------
# import glob
# import numpy as np
# import librosa

# segment_dur = 2
# directories = ['guitar', 'non_guitar']
# def load_segments(directory):
#     segments = []
#     for path in glob.glob(f'{directory}/*.wav'):
#         signal, sr = librosa.load(path, sr=None, mono=True)
#         seg_len = sr * segment_dur
#         audio_segments = librosa.util.frame(signal, frame_length=seg_len, hop_length=seg_len, axis=0)
#         segments.append(audio_segments)
#     return np.concatenate(segments)

# guitar_segments = load_segments('guitar')
# non_guitar_segments = load_segments('non_guitar')
# def get_chords(path):

#     chords = []
#     with open(path) as f:
#         for line in f:      
#             start_s, end_s, chord = line.strip().split('\t')
#             chords.append(chord)
#     return chords


# guitar_chords = []
# for i in range(6):
#     guitar_chords += get_chords(f'guitar/guitar_annotation.lab')

# non_guitar_chords = []
# for i in range(10):
#     non_guitar_chords += get_chords(f'non_guitar/non_guitar_annotation.lab')

using mps


In [2]:
#Everything needed to use a trained model (also for the presentation)
import torch

#set up device
device = "mps" if torch.backends.mps.is_available() else "cpu" #mps is the Mac's GPU
print("using", device)


#sklearn: returns predicted label and confidence
def predict_sklearn_model(model, data):
    labels = model.predict(data)
    confidences = None #a plain SVM cant give confidences
    if hasattr(model, "predict_proba"):
        confidences = model.predict_proba(data).max(axis=1)
    return labels, confidences


#MLP and CNN: scales the data and turns it into a tensor
def get_inputs(data, scaler):
    if scaler is not None:
        data = scaler.transform(data)
    return torch.tensor(data, dtype=torch.float32).to(device)


#MLP and CNN: returns predicted label and confidence
def predict_net(net, data):
    inputs = get_inputs(data, net["scaler"])
    net["model"].eval() #testing mode
    with torch.no_grad(): #not learning, so dont keep track of anything
        scores = net["model"](inputs)
    probabilities = torch.softmax(scores, dim=1).cpu().numpy() #each clip's probabilities add up to 1
    labels = net["classes"][probabilities.argmax(axis=1)]
    confidences = probabilities.max(axis=1)
    return labels, confidences


#Combines root and quality labels
def combine_labels(roots, qualities):
    return [root + ":" + quality for root, quality in zip(roots, qualities)]

using mps


In [3]:
#Set up and helper functions for training
import copy
import librosa
import numpy as np
import pandas as pd
import torch.nn as nn

from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, Normalizer, StandardScaler
from sklearn.svm import SVC

#Helpers

#sklearn: makes a fresh untrained model with its best tuned settings
def build_sklearn_model(name, target):
    model = clone(sklearn_models[name])
    return model.set_params(**best_params.get((name, target), {}))


#MLP and CNN: picks the features, the CNN uses the picture, the MLP uses X
def get_data(name):
    return picture if nets[name]["input"] == "picture" else X


#Set up

#Generate labels

#True keeps the inversions (288 chords), False drops them (84 chords)
INCLUDE_INVERSIONS = True 
labels = pd.read_csv("data/processed/labels.csv")

if INCLUDE_INVERSIONS:
    y = labels["original_label"].to_numpy()
else:
    y = labels["original_label"].str.split("/").str[0].to_numpy()


#Set up targets. Quality includes inversions.
targets = {
    "chord": y,
    "root": np.array([label.split(":")[0] for label in y]),
    "quality": np.array([label.split(":")[1] for label in y]),
}

for name, values in targets.items():
    print(name, len(set(values)), "classes")

chord 288 classes
root 12 classes
quality 24 classes


In [ ]:
#SPLIT OPTION 1: random (70% train, 15% validation, 15% test)
#only run one of the 2 split cells

all_idx = np.arange(len(y))
train_idx, rem_idx = train_test_split(all_idx, test_size=0.3, stratify=y, random_state=42)
val_idx, test_idx = train_test_split(rem_idx, test_size=0.5, stratify=y[rem_idx], random_state=42)

split_used = "random"
print(split_used, len(train_idx), len(val_idx), len(test_idx))

[   0    1    2 ... 7395 7396 7397]
random 5178 1110 1110


In [ ]:
#SPLIT OPTION 2: by instrument (11 train, 2 validation, 3 test)

VAL_INSTRUMENTS = [4, 10]
TEST_INSTRUMENTS = [3, 6, 13]

instruments = labels["instrument"].unique()


val_mask = []
for val_instrument in VAL_INSTRUMENTS:
    val_mask.append(labels["instrument"] == instruments[val_instrument])
val_mask = np.array(val_mask).any(axis=0)

test_mask = []
for test_instrument in TEST_INSTRUMENTS:
    test_mask.append(labels["instrument"] == instruments[test_instrument])
test_mask = np.array(test_mask).any(axis=0)

train_idx = np.where(~val_mask & ~test_mask)[0]
val_idx = np.where(val_mask)[0]
test_idx = np.where(test_mask)[0]


split_used = "instrument"
print(split_used, len(train_idx), len(val_idx), len(test_idx))

instrument 5124 849 1425


In [ ]:
#FEATURES OPTION 1: chroma
#only run one of the 2 features cells

sr = 16000
X_audio = np.load("data/processed/clips_16k.npy").astype(np.float32)
picture = librosa.feature.chroma_stft(y=X_audio, sr=sr, n_fft=8192, hop_length=4096) # 7398 chords, 12 bins, 8 time frames
X = np.concatenate([picture.mean(axis=-1), picture.std(axis=-1)], axis=1) # 24 features: Mean and std of each of the 12 chroma bins

features_used = "chroma"
print(features_used, X.shape, picture.shape)

(7398, 32000)
chroma (7398, 24) (7398, 12, 8)


In [7]:
#FEATURES OPTION 2: wide spectrum (keeps the octaves separate, so inversions can be told apart)

sr = 16000
X_audio = np.load("data/processed/clips_16k.npy").astype(np.float32)

# Take abs of cqt to get magnitude of complex number (only want strength not phase)
cqt = np.abs(librosa.cqt(y=X_audio, sr=sr, hop_length=4096, fmin=librosa.note_to_hz("C1"), n_bins=84)) #84 notes, 8 time frames
picture = cqt / cqt.max(axis=(1, 2), keepdims=True) # 7398 chords, 84 bins, 8 time frames
X = X = np.concatenate([picture.mean(axis=-1), picture.std(axis=-1)], axis=1)
features_used = "wide spectrum"
print(features_used, X.shape, picture.shape)

wide spectrum (7398, 168) (7398, 84, 8)


In [46]:
#the starting settings for the sklearn models

#each model gets its own scaler, which only ever learns from the training data
sklearn_models = {
    "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=500))]),
    "k-NN": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier())]),
    "SVM": Pipeline([("scaler", StandardScaler()), ("model", SVC(C=10))]),
    "Random Forest": Pipeline([("scaler", StandardScaler()), ("model", RandomForestClassifier(random_state=42))]),
}

In [ ]:
#sklearn tuning: tries every combination in each grid and keeps whichever does best on the validation set

# ADD EVALUATION METRICS AND GRAPH STUFF HERE:

SCALERS = ["passthrough", StandardScaler(), MinMaxScaler(), Normalizer()] #passthrough means no scaling
grids = {
    "Logistic Regression": {"scaler": SCALERS, "model__C": [0.1, 1, 10]},
    "k-NN": {"scaler": SCALERS, "model__n_neighbors": [1, 3, 5, 9, 15, 25, 50, 75, 100], "model__metric": ["euclidean", "manhattan", "cosine"]},
    "SVM": {"scaler": SCALERS, "model__kernel": ["linear", "rbf", "poly"], "model__C": [0.1, 1, 10,100]},
    "Random Forest": {"scaler": ["passthrough", Normalizer()], "model__n_estimators": [100, 200, 300, 400, 500], "model__min_samples_leaf": [1, 3, 5, 10]},
}

best_params = {} 

for target in targets:
    for name, grid in grids.items():
        search = GridSearchCV(sklearn_models[name], grid, cv=[(train_idx, val_idx)], n_jobs=-1)
        search.fit(X, targets[target])
        best_params[name, target] = search.best_params_
        print(target, name, search.best_params_)

chord, Logistic Regression: val accuracy 0.402 {'model__C': 1, 'scaler': StandardScaler()}
    runner up 0.396 {'model__C': 10, 'scaler': MinMaxScaler()}
    runner up 0.393 {'model__C': 10, 'scaler': 'passthrough'}
    runner up 0.382 {'model__C': 10, 'scaler': Normalizer()}
chord, k-NN: val accuracy 0.519 {'model__metric': 'euclidean', 'model__n_neighbors': 1, 'scaler': Normalizer()}
    runner up 0.519 {'model__metric': 'euclidean', 'model__n_neighbors': 1, 'scaler': Normalizer()}
    runner up 0.519 {'model__metric': 'cosine', 'model__n_neighbors': 1, 'scaler': Normalizer()}
    runner up 0.435 {'model__metric': 'manhattan', 'model__n_neighbors': 1, 'scaler': Normalizer()}
chord, SVM: val accuracy 0.538 {'model__C': 100, 'model__kernel': 'rbf', 'scaler': Normalizer()}
    runner up 0.534 {'model__C': 10, 'model__kernel': 'rbf', 'scaler': Normalizer()}
    runner up 0.523 {'model__C': 10, 'model__kernel': 'poly', 'scaler': Normalizer()}
    runner up 0.516 {'model__C': 100, 'model__

/Users/joanmk/UniStuff/IAML/IDMT-SMT-CHORDS/.venv/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:787: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


chord, Random Forest: val accuracy 0.516 {'model__min_samples_leaf': 3, 'model__n_estimators': 500, 'scaler': 'passthrough'}
    runner up 0.514 {'model__min_samples_leaf': 3, 'model__n_estimators': 400, 'scaler': 'passthrough'}
    runner up 0.511 {'model__min_samples_leaf': 1, 'model__n_estimators': 500, 'scaler': 'passthrough'}
    runner up 0.509 {'model__min_samples_leaf': 3, 'model__n_estimators': 300, 'scaler': 'passthrough'}
root, Logistic Regression: val accuracy 0.779 {'model__C': 10, 'scaler': Normalizer()}
    runner up 0.759 {'model__C': 10, 'scaler': 'passthrough'}
    runner up 0.757 {'model__C': 10, 'scaler': MinMaxScaler()}
    runner up 0.730 {'model__C': 0.1, 'scaler': StandardScaler()}
root, k-NN: val accuracy 0.726 {'model__metric': 'euclidean', 'model__n_neighbors': 1, 'scaler': Normalizer()}
    runner up 0.726 {'model__metric': 'cosine', 'model__n_neighbors': 1, 'scaler': Normalizer()}
    runner up 0.726 {'model__metric': 'cosine', 'model__n_neighbors': 1, 'sca

In [ ]:
#sklearn models, single model

print(features_used, "features,", split_used, "split")
for name in sklearn_models:
    model = build_sklearn_model(name, "chord")
    model.fit(X[train_idx], y[train_idx])
    val_accuracy = model.score(X[val_idx], y[val_idx])

    print(f"{name}: val {val_accuracy:.3f}")

wide spectrum features, random split
Logistic Regression: val 0.336
k-NN: val 0.166
SVM: val 0.485


In [47]:
#sklearn models, double model
print(features_used, "features,", split_used, "split")
for name in sklearn_models:
    root_model = build_sklearn_model(name, "root")
    quality_model = build_sklearn_model(name, "quality")
    root_model.fit(X[train_idx], targets["root"][train_idx])
    quality_model.fit(X[train_idx], targets["quality"][train_idx])

    root_pred, root_confidence = predict_sklearn_model(root_model, X[val_idx])
    quality_pred, quality_confidence = predict_sklearn_model(quality_model, X[val_idx])
    combined_pred = combine_labels(root_pred, quality_pred)

    print(name)
    print(f"root accuracy: {accuracy_score(targets['root'][val_idx], root_pred):.3f}")
    print(f"quality accuracy: {accuracy_score(targets['quality'][val_idx], quality_pred):.3f}")
    print(f"combined: {accuracy_score(y[val_idx], combined_pred):.3f}\n")

wide spectrum features, random split
Logistic Regression
root accuracy: 0.843
quality accuracy: 0.090
combined: 0.067

k-NN
root accuracy: 0.662
quality accuracy: 0.189
combined: 0.166

SVM
root accuracy: 0.886
quality accuracy: 0.313
combined: 0.278

Random Forest
root accuracy: 0.817
quality accuracy: 0.158
combined: 0.119



In [ ]:
#shared code for the MLP and CNN

MAX_EPOCHS = 100
PATIENCE = 30 #stops early if the validation accuracy hasnt improved for this many epochs
SEED = 42

nets = {} #the MLP and CNN cells add themselves to this
predictions = {} #filled in by the 3 prediction cells


#trains one model
def train_net(name, target, settings):
    torch.manual_seed(SEED)

    #1. get the data ready
    data = get_data(name)
    scaler = settings["scaler"]
    if scaler is not None:
        scaler = clone(scaler).fit(data[train_idx]) #only learns from the training data
    inputs = get_inputs(data[train_idx], scaler)
    classes, numbers = np.unique(targets[target], return_inverse=True) #gives each label a number
    answers = torch.tensor(numbers[train_idx]).to(device)

    #2. make the model
    model = nets[name]["make"](inputs.shape[1:], len(classes), settings).to(device)
    loss_function = nn.CrossEntropyLoss() #measures how wrong the predictions are
    optimizer = torch.optim.Adam(model.parameters(), lr=settings["lr"], weight_decay=settings["weight_decay"])
    net = {"model": model, "scaler": scaler, "classes": classes, "val_accuracy": -1}

    epochs_since_best = 0
    for epoch in range(MAX_EPOCHS):
        #3. learn from the training data, one shuffled batch at a time
        model.train()
        for batch in torch.randperm(len(inputs)).split(settings["batch_size"]):
            optimizer.zero_grad() #forgets the last batch
            loss = loss_function(model(inputs[batch]), answers[batch]) #how wrong was it
            loss.backward() #works out how to change each weight
            optimizer.step() #changes them

        #4. check the validation accuracy, and remember the weights if its the best so far
        labels, confidences = predict_net(net, data[val_idx])
        accuracy = accuracy_score(targets[target][val_idx], labels)
        if accuracy > net["val_accuracy"]:
            net["val_accuracy"] = accuracy
            best_weights = copy.deepcopy(model.state_dict())
            epochs_since_best = 0
        else:
            epochs_since_best += 1
        if epochs_since_best == PATIENCE:
            break

    model.load_state_dict(best_weights) #goes back to the best epoch
    return net


#trains one model with its current settings and gives its validation predictions
def run_net(name, target):
    net = train_net(name, target, nets[name]["settings"][target])
    nets[name]["trained"][target] = net
    print(f"{name} {target}: val {net['val_accuracy']:.3f}")
    labels, confidences = predict_net(net, get_data(name)[val_idx])
    return labels


#changes one setting at a time and keeps the change if the validation accuracy improves

# ADD EVALUATION GRAPHS FOR THESE
def tune_net(name, target):
    best_settings = nets[name]["settings"][target]
    best_accuracy = train_net(name, target, best_settings)["val_accuracy"]

    for setting, options in nets[name]["options"].items():
        for option in options:
            new_settings = {**best_settings, setting: option} #a copy with one setting changed
            accuracy = train_net(name, target, new_settings)["val_accuracy"]
            if accuracy > best_accuracy:
                best_accuracy = accuracy
                best_settings = new_settings

    nets[name]["settings"][target] = best_settings
    print(target, name, best_settings)

In [ ]:
#the MLP
def make_mlp(input_shape, n_classes, settings):
    layers = []
    size = input_shape[0]
    for hidden in settings["hidden"]:
        layers += [nn.Linear(size, hidden), nn.ReLU(), nn.Dropout(settings["dropout"])]
        size = hidden
    layers.append(nn.Linear(size, n_classes))
    return nn.Sequential(*layers)


mlp_settings = {"scaler": StandardScaler(), "hidden": [128, 128], "dropout": 0, "lr": 0.001, "weight_decay": 0, "batch_size": 64}

#what the tuning cell tries
mlp_options = {
    "scaler": [None, MinMaxScaler(), Normalizer()],
    "hidden": [[128], [256, 256], [512, 512], [256, 256, 256]],
    "dropout": [0.2, 0.4],
    "lr": [0.0003, 0.003],
    "weight_decay": [0.0001, 0.001],
    "batch_size": [32, 128],
}

nets["MLP"] = {
    "make": make_mlp,
    "input": "features",
    "settings": {target: dict(mlp_settings) for target in targets},
    "options": mlp_options,
    "trained": {},
}

In [ ]:
#the CNN
def make_cnn(input_shape, n_classes, settings):
    notes, frames = input_shape
    kernel = settings["kernel"]
    layers = [nn.Unflatten(1, (1, notes))] #adds the channel dimension
    channels_in = 1
    for channels in settings["channels"]:
        layers.append(nn.Conv2d(channels_in, channels, kernel_size=kernel, padding=kernel // 2))
        if settings["batchnorm"]:
            layers.append(nn.BatchNorm2d(channels))
        layers += [nn.ReLU(), nn.MaxPool2d((1, 2))] #only pools over time, so the notes stay separate
        channels_in = channels
        frames = frames // 2
    layers += [
        nn.Flatten(),
        nn.Dropout(settings["dropout"]),
        nn.Linear(channels_in * notes * frames, settings["hidden"]),
        nn.ReLU(),
        nn.Dropout(settings["dropout"]),
        nn.Linear(settings["hidden"], n_classes),
    ]
    return nn.Sequential(*layers)


cnn_settings = {"scaler": None, "channels": [16, 32], "kernel": 3, "batchnorm": False, "hidden": 128, "dropout": 0, "lr": 0.001, "weight_decay": 0, "batch_size": 64}

#what the tuning cell tries (3 pools is the most that fits, because there are only 8 time frames)
cnn_options = {
    "channels": [[32, 64], [16, 32, 64]],
    "kernel": [5],
    "batchnorm": [True],
    "hidden": [256],
    "dropout": [0.3, 0.5],
    "lr": [0.0003, 0.003],
    "weight_decay": [0.0001, 0.001],
    "batch_size": [32, 128],
}

nets["CNN"] = {
    "make": make_cnn,
    "input": "picture",
    "settings": {target: dict(cnn_settings) for target in targets},
    "options": cnn_options,
    "trained": {},
}

In [ ]:
#MLP and CNN tuning

# ADD EVALUATION GRAPHS FOR THESE (CODE FOR TUNING NET IS ABOVE)
RUN_NET_TUNING = False
TUNE_NETS = ["MLP", "CNN"]
TUNE_TARGETS = ["chord", "root", "quality"]

if RUN_NET_TUNING:
    for name in TUNE_NETS:
        for target in TUNE_TARGETS:
            tune_net(name, target)

In [ ]:
#predicts the whole chord in one go
for name in nets:
    predictions[name, "chord"] = run_net(name, "chord")

In [ ]:
#predicts just the root
for name in nets:
    predictions[name, "root"] = run_net(name, "root")

In [ ]:
#predicts just the quality
for name in nets:
    predictions[name, "quality"] = run_net(name, "quality")

In [ ]:
#combines the root and quality predictions into full chords
for name in nets:
    root_pred = predictions[name, "root"]
    quality_pred = predictions[name, "quality"]
    combined_pred = combine_labels(root_pred, quality_pred)

    print(name)
    print(f"root accuracy: {accuracy_score(targets['root'][val_idx], root_pred):.3f}")
    print(f"quality accuracy: {accuracy_score(targets['quality'][val_idx], quality_pred):.3f}")
    print(f"combined: {accuracy_score(y[val_idx], combined_pred):.3f}")
    print(f"whole chord in one go: {accuracy_score(y[val_idx], predictions[name, 'chord']):.3f}\n")